<h3>Runnable & Runnable Passthrough</h3>

Runnable is anything you can “pipe” (|) data through (prompts, LLMs, retrievers, functions, etc.).
RunnablePassthrough is a tiny helper Runnable that forwards its input unchanged. It’s most useful when you want to:

Preserve the original input while also computing extra fields derived from that input.
Branch: feed the same input to multiple sub-steps (e.g., to an LLM and to a function) and then recombine.
Enrich: add keys to an object (dict) on-the-fly without losing existing keys.

Think of it as “keep the input as-is, but let me attach more stuff next to it.”
There are two very common patterns:

RunnablePassthrough() — just forwards input.
RunnablePassthrough.assign(key1=..., key2=...) — forwards input and adds/overwrites fields using other runnables/functions.

There are two very common patterns:

RunnablePassthrough() — just forwards input.
RunnablePassthrough.assign(key1=..., key2=...) — forwards input and adds/overwrites fields using other runnables/functions.

In [ ]:
# Simple passthrough runnable example
from langchain_core.runnables import RunnablePassthrough

pt = RunnablePassthrough()

print(pt.invoke("Hello"))       # -> "Hello"
print(pt.invoke({"x": 1}))      # -> {"x": 1}


Hello
{'x': 1}


In [2]:
# Example of a more complex runnable that adds 1 to a number
from langchain_core.runnables import Runnable

class AddOneRunnable(Runnable):
    def invoke(self, input):
        return input + 1

add_one = AddOneRunnable()
print(add_one.invoke(5))  # -> 6

6


In [3]:
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0, max_tokens=500)

def extract_city(user_text: str) -> str:
    for city in ["Paris", "Riyadh", "Jeddah", "Lahore"]:
        if city.lower() in user_text.lower():
            return city
    return "Unknown"

# 1) Build a prompt that expects both 'question' and 'city'
prompt = ChatPromptTemplate.from_template(
    "Answer the user's question.\n"
    "City (detected): {city}\n"
    "Question: {question}"
)

# 2) Build a small preprocessor that turns the input string into a dict {question: <str>} and enriches it
pre = (
    # Turn the raw string into a mapping with a 'question' key
    (lambda x: {"question": x})
    # Add 'city' while preserving 'question'
    | RunnablePassthrough.assign(city=lambda d: extract_city(d["question"]))
)

# 3) Compose the full chain
chain = pre | prompt | llm

print(chain.invoke("What's the weather in Paris today?").content)

### We'll fix the above issue with agents, where we can do weather search and bring the results back as part of results!

In [4]:
# Another RunnablePassthrough example: enrich input with derived fields
enrich_chain = (
    (lambda text: {"text": text})
    | RunnablePassthrough.assign(
        char_count=lambda d: len(d["text"]),
        word_count=lambda d: len(d["text"].split()),
        is_question=lambda d: d["text"].strip().endswith("?"),
        uppercase=lambda d: d["text"].upper(),
    )
)

result = enrich_chain.invoke("Can RunnablePassthrough enrich this input?")
print(result)

{'text': 'Can RunnablePassthrough enrich this input?', 'char_count': 42, 'word_count': 5, 'is_question': True, 'uppercase': 'CAN RUNNABLEPASSTHROUGH ENRICH THIS INPUT?'}
